In [4]:
!pip install dagshub mlflow catboost -q
import dagshub
import mlflow
import pandas as pd
import numpy as np
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
token = user_secrets.get_secret("DAGSHUB_TOKEN")

dagshub.init(repo_owner='kvizhmena11', repo_name='ML-assignment2-IEEE-Fraud-Detection', mlflow=True)
mlflow.set_tracking_uri("https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow")
mlflow.set_experiment("CatBoost_Training")
print("✅ Connected to DagsHub!")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=01a9487e-4b11-47b8-9bd2-deab76e1cd7e&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=55199c893cb51918a9ce8e195238b6feaa715b3e1542ab9863157276705e8e94




Accessing as kvizhmena11

Initialized MLflow to track repo "kvizhmena11/ML-assignment2-IEEE-Fraud-Detection"

Repository kvizhmena11/ML-assignment2-IEEE-Fraud-Detection initialized!

✅ Connected to DagsHub!


# 🧹 Cleaning


In [5]:
import os
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score

def reduce_mem_usage(df, verbose=True):
    numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
    start_mem = df.memory_usage().sum() / 1024**2    
    for col in df.columns:
        col_type = df[col].dtype
        if col_type in numerics:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                else:
                    df[col] = df[col].astype(np.int64)  
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)    
    end_mem = df.memory_usage().sum() / 1024**2
    if verbose: print('Mem. usage decreased to {:5.2f} Mb ({:.1f}% reduction)'.format(end_mem, 100 * (start_mem - end_mem) / start_mem))
    return df

mlflow.end_run()
base_path = '/kaggle/input/competitions/ieee-fraud-detection/'

with mlflow.start_run(run_name="CatBoost_Cleaning"):
    train_transaction = pd.read_csv(os.path.join(base_path, 'train_transaction.csv'))
    train_identity = pd.read_csv(os.path.join(base_path, 'train_identity.csv'))
    
    train = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')
    del train_transaction, train_identity
    
    nan_count = train.isna().sum().sum()
    fraud_rate = train['isFraud'].mean()
    
    mlflow.log_metric("initial_nan_count", nan_count)
    mlflow.log_metric("fraud_rate_pct", fraud_rate * 100)
    mlflow.log_metric("class_imbalance_ratio", (1 - fraud_rate) / fraud_rate)
    
    train = reduce_mem_usage(train)
    
    mlflow.log_param("total_rows", len(train))
    mlflow.log_param("total_columns", len(train.columns))
    mlflow.log_param("nan_strategy", "keep_as_nan_catboost_handles_natively")
    print(f"✅ Dataset shape: {train.shape}")
    print(f"Fraud rate: {fraud_rate:.4f}")

Mem. usage decreased to 1044.70 Mb (46.6% reduction)
✅ Dataset shape: (590540, 434)
Fraud rate: 0.0350
🏃 View run CatBoost_Cleaning at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/2/runs/0413358205d348938b49313f2a3b2a74
🧪 View experiment at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/2


# ⚙️ Feature Engineering


In [12]:
with mlflow.start_run(run_name="CatBoost_Feature_Engineering"):
    
    emails = {'gmail': 'google', 'att.net': 'at&t', 'twc.com': 'spectrum',
              'outlook.com': 'microsoft', 'msn.com': 'microsoft', 'hotmail.com': 'microsoft',
              'live.com': 'microsoft', 'icloud.com': 'apple', 'mac.com': 'apple'}
    
    for c in ['P_emaildomain', 'R_emaildomain']:
        train[c + '_bin'] = train[c].map(emails)
        train[c + '_suffix'] = train[c].map(lambda x: str(x).split('.')[-1])
    
    train['Trans_hour'] = (train['TransactionDT'] // 3600) % 24
    train['Trans_day_of_week'] = (train['TransactionDT'] // (3600 * 24)) % 7
    
    for col in ['card1', 'card2', 'card3', 'card5']:
        if col in train.columns:
            freq = train[col].astype(np.float32).value_counts(normalize=True)
            train[f'{col}_freq'] = train[col].astype(np.float32).map(freq)
    
    train['TransactionAmt_log'] = np.log1p(train['TransactionAmt'])
    train['TransactionAmt_decimal'] = ((train['TransactionAmt'].astype(np.float32) - train['TransactionAmt'].astype(int)) * 1000).astype(int)
    
    train['card1_TransactionAmt_mean'] = train.groupby(train['card1'].astype(np.float32))['TransactionAmt'].transform('mean')
    train['card1_TransactionAmt_std'] = train.groupby(train['card1'].astype(np.float32))['TransactionAmt'].transform('std')
    train['amt_vs_card_mean'] = train['TransactionAmt'] / (train['card1_TransactionAmt_mean'] + 1)
    train['TransactionAmt_zscore'] = (train['TransactionAmt'] - train['card1_TransactionAmt_mean']) / (train['card1_TransactionAmt_std'] + 1)
    
    if 'addr1' in train.columns and 'addr2' in train.columns:
        train['addr_match'] = (train['addr1'] == train['addr2']).astype(int)
    
    from sklearn.preprocessing import LabelEncoder
    cat_cols = ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3',
                'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23',
                'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35',
                'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo',
                'P_emaildomain_bin', 'P_emaildomain_suffix', 'R_emaildomain_bin', 'R_emaildomain_suffix']
    
    for col in cat_cols:
        if col in train.columns:
            le = LabelEncoder()
            train[col] = le.fit_transform(train[col].astype(str))
    
    train = train.copy()
    
    mlflow.log_param("total_engineered_features", 8)
    mlflow.log_param("encoding_strategy", "label_encoding_no_cat_features")
    print(f"✅ Shape after FE: {train.shape}")

✅ Shape after FE: (590540, 441)
🏃 View run CatBoost_Feature_Engineering at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/2/runs/8d61f3e7986b403a9e52de39d27ad109
🧪 View experiment at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/2


# 🎯 Feature Selection


In [13]:
with mlflow.start_run(run_name="CatBoost_Feature_Selection"):
    
    threshold = 0.90
    null_cols = [c for c in train.columns if train[c].isnull().sum() / len(train) > threshold]
    one_value_cols = [c for c in train.columns if train[c].nunique() <= 1]
    cols_to_drop = list(set(null_cols + one_value_cols))
    for c in ['isFraud', 'TransactionID', 'TransactionDT']:
        if c in cols_to_drop:
            cols_to_drop.remove(c)
    train.drop(cols_to_drop, axis=1, inplace=True)
    mlflow.log_param("cols_dropped_null_or_constant", len(cols_to_drop))
    
    cat_cols = [c for c in cat_cols if c in train.columns]
    
    X_fs = train.drop(['isFraud', 'TransactionID', 'TransactionDT'], axis=1, errors='ignore')
    y_fs = train['isFraud']
    
    sample_idx = X_fs.sample(frac=0.1, random_state=42).index
    X_sample = X_fs.loc[sample_idx]
    y_sample = y_fs.loc[sample_idx]
    
    cat_feature_indices = [X_fs.columns.tolist().index(c) for c in cat_cols if c in X_fs.columns]
    
    quick_model = CatBoostClassifier(
        n_estimators=100, max_depth=6, random_state=42, verbose=0
    )
    quick_model.fit(X_sample, y_sample, cat_features=cat_feature_indices)
    
    importances = pd.Series(quick_model.feature_importances_, index=X_fs.columns)
    selected_features = importances[importances > 0].index.tolist()
    removed_low_importance = len(X_fs.columns) - len(selected_features)
    
    cat_cols = [c for c in cat_cols if c in selected_features]
    cat_feature_indices = [selected_features.index(c) for c in cat_cols]
    
    mlflow.log_param("selection_method", "catboost_importance_on_10pct_sample")
    mlflow.log_param("features_removed_low_importance", removed_low_importance)
    mlflow.log_param("final_selected_features", len(selected_features))
    mlflow.log_param("final_cat_cols_count", len(cat_cols))
    
    print(f"✅ Selected {len(selected_features)} features out of {len(X_fs.columns)}")
    print(f"Categorical features remaining: {len(cat_cols)}")

✅ Selected 238 features out of 437
Categorical features remaining: 24
🏃 View run CatBoost_Feature_Selection at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/2/runs/01f0fc745ed6411ca0a65f75b4d72643
🧪 View experiment at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/2


# 🚀 Training


In [14]:
X = train[selected_features]
y = train['isFraud']
split_idx = int(len(X) * 0.8)
X_train, X_val = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_val = y.iloc[:split_idx], y.iloc[split_idx:]
print(f"Train: {len(X_train)} | Val: {len(X_val)}")

Train: 472432 | Val: 118108


In [16]:
with mlflow.start_run(run_name="CatBoost_Underfit"):
    params = {'iterations': 50, 'depth': 2, 'learning_rate': 0.3,
              'random_seed': 42, 'verbose': 0}
    mlflow.log_params(params)
    mlflow.log_param("experiment_purpose", "underfitting_demonstration")
    
    m = CatBoostClassifier(**params)
    m.fit(X_train, y_train)    
    train_auc = roc_auc_score(y_train, m.predict_proba(X_train)[:,1])
    val_auc = roc_auc_score(y_val, m.predict_proba(X_val)[:,1])
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("val_auc", val_auc)
    mlflow.log_metric("overfit_gap", train_auc - val_auc)
    print(f"[UNDERFIT] Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {train_auc-val_auc:.4f}")

[UNDERFIT] Train: 0.8685 | Val: 0.8527 | Gap: 0.0159
🏃 View run CatBoost_Underfit at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/2/runs/080d553cd73c4c07996706d727de98c6
🧪 View experiment at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/2


In [17]:
with mlflow.start_run(run_name="CatBoost_Overfit"):
    params = {'iterations': 300, 'depth': 10, 'learning_rate': 0.3,
              'random_seed': 42, 'verbose': 50}
    mlflow.log_params(params)
    mlflow.log_param("experiment_purpose", "overfitting_demonstration")
    m = CatBoostClassifier(**params)
    m.fit(X_train, y_train)
    train_auc = roc_auc_score(y_train, m.predict_proba(X_train)[:,1])
    val_auc = roc_auc_score(y_val, m.predict_proba(X_val)[:,1])
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("val_auc", val_auc)
    mlflow.log_metric("overfit_gap", train_auc - val_auc)
    print(f"[OVERFIT] Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {train_auc-val_auc:.4f}")

0:	learn: 0.3113118	total: 861ms	remaining: 4m 17s
50:	learn: 0.0680166	total: 42.9s	remaining: 3m 29s
100:	learn: 0.0558212	total: 1m 24s	remaining: 2m 47s
150:	learn: 0.0474525	total: 2m 6s	remaining: 2m 5s
200:	learn: 0.0409032	total: 2m 48s	remaining: 1m 23s
250:	learn: 0.0363530	total: 3m 30s	remaining: 41.2s
299:	learn: 0.0324855	total: 4m 12s	remaining: 0us
[OVERFIT] Train: 0.9883 | Val: 0.8941 | Gap: 0.0943
🏃 View run CatBoost_Overfit at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/2/runs/ecd90c1df3834345b5f1141aae5139d1
🧪 View experiment at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/2


In [18]:
with mlflow.start_run(run_name="CatBoost_Best"):
    params = {'iterations': 300, 'depth': 6, 'learning_rate': 0.05,
              'subsample': 0.7, 'colsample_bylevel': 0.7,
              'reg_lambda': 5.0, 'min_data_in_leaf': 20,
              'scale_pos_weight': 30, 'random_seed': 42, 'verbose': 50}
    mlflow.log_params(params)
    mlflow.log_param("experiment_purpose", "best_balanced_model")
    m = CatBoostClassifier(**params)
    m.fit(X_train, y_train)  # no cat_features
    train_auc = roc_auc_score(y_train, m.predict_proba(X_train)[:,1])
    val_auc = roc_auc_score(y_val, m.predict_proba(X_val)[:,1])
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("val_auc", val_auc)
    mlflow.log_metric("overfit_gap", train_auc - val_auc)
    print(f"[BEST] Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {train_auc-val_auc:.4f}")

0:	learn: 0.6704488	total: 180ms	remaining: 53.8s
50:	learn: 0.4431051	total: 9.46s	remaining: 46.2s
100:	learn: 0.4116649	total: 18.7s	remaining: 36.9s
150:	learn: 0.3929206	total: 27.9s	remaining: 27.5s
200:	learn: 0.3803726	total: 36.9s	remaining: 18.2s
250:	learn: 0.3687887	total: 45.8s	remaining: 8.95s
299:	learn: 0.3547195	total: 54.8s	remaining: 0us
[BEST] Train: 0.9253 | Val: 0.8997 | Gap: 0.0256
🏃 View run CatBoost_Best at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/2/runs/e0b8db6029ca4d179a23e5a0ff2fc34c
🧪 View experiment at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/2


# 💾 Pipeline & Model Registry

In [19]:
import mlflow.sklearn
import json
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

best_params = {
    'iterations': 300, 'depth': 6, 'learning_rate': 0.05,
    'subsample': 0.7, 'colsample_bylevel': 0.7,
    'reg_lambda': 5.0, 'min_data_in_leaf': 20,
    'scale_pos_weight': 30, 'random_seed': 42, 'verbose': 0
}

best_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value=-999)),
    ('classifier', CatBoostClassifier(**best_params))
])

best_pipeline.fit(X_train, y_train)

with mlflow.start_run(run_name="CatBoost_Final_Pipeline"):
    train_auc = roc_auc_score(y_train, best_pipeline.predict_proba(X_train)[:,1])
    val_auc = roc_auc_score(y_val, best_pipeline.predict_proba(X_val)[:,1])
    
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("val_auc", val_auc)
    mlflow.log_metric("overfit_gap", train_auc - val_auc)
    mlflow.log_param("selected_features_count", len(selected_features))
    
    mlflow.sklearn.log_model(
        sk_model=best_pipeline,
        artifact_path="catboost_pipeline",
        registered_model_name="CatBoost_Fraud_Pipeline",
        input_example=X_val.iloc[:5]
    )
    
    print(f"✅ Pipeline saved to Model Registry!")
    print(f"Train AUC: {train_auc:.4f} | Val AUC: {val_auc:.4f}")

with open('selected_features_catboost.json', 'w') as f:
    json.dump(selected_features, f)
mlflow.log_artifact('selected_features_catboost.json')

mlflow.end_run()
print("✅ CatBoost notebook complete!")

2026/05/07 12:19:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 12:19:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can 

✅ Pipeline saved to Model Registry!
Train AUC: 0.9256 | Val AUC: 0.9009
🏃 View run CatBoost_Final_Pipeline at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/2/runs/b36ec70cafcc4e2288b13c818c52cdfa
🧪 View experiment at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/2
🏃 View run sassy-shrike-472 at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/2/runs/3e602f9df9494e9fb5c8ff0ed4771925
🧪 View experiment at: https://dagshub.com/kvizhmena11/ML-assignment2-IEEE-Fraud-Detection.mlflow/#/experiments/2
✅ CatBoost notebook complete!
